In [8]:
"""
Word Alignment Comparison: SimAlign vs AwesomeAlign vs fast_align vs WFV (CombAlign)
Dataset : English--Tamil sentence pairs
         (Corrected_Tamil_Alignments_1-134.xlsx)
Mirrors the Hindi comparison pipeline exactly, adapted for Tamil WFV logic.
"""

# ─────────────────────────────────────────────
# 0. AUTO-INSTALL / AUTO-BUILD DEPENDENCIES
# ─────────────────────────────────────────────
import subprocess, sys, os

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])

def shell(cmd):
    subprocess.check_call(cmd, shell=True)

pip_install("simalign")
pip_install("transformers", "torch", "numpy", "scikit-learn", "scipy")
pip_install("sentence-transformers")
pip_install("pandas", "openpyxl")
pip_install("git+https://github.com/neulab/awesome-align.git")

# ── Build fast_align C++ binary (one-time, ~1 min) ──────────────────────────
FA_BIN  = "/tmp/fast_align_build/fast_align"
AGG_BIN = "/tmp/fast_align_build/atools"

if not os.path.isfile(FA_BIN):
    print("⏳ Building fast_align from source (one-time, ~1 min) …")
    shell("apt-get install -yq build-essential cmake libgoogle-perftools-dev "
          "2>/dev/null || true")
    shell(
        "rm -rf /tmp/fa_src && "
        "git clone --depth 1 https://github.com/clab/fast_align /tmp/fa_src && "
        "mkdir -p /tmp/fast_align_build && "
        "cmake -S /tmp/fa_src -B /tmp/fast_align_build "
        "      -DCMAKE_BUILD_TYPE=Release -DGPERFTOOLS_FOUND=OFF && "
        "cmake --build /tmp/fast_align_build --parallel 4"
    )
    print("✅ fast_align built.")
else:
    print("✅ fast_align binary already present.")


# ─────────────────────────────────────────────
# 1. LOAD DATASET FROM EXCEL
# ─────────────────────────────────────────────
import ast
import pandas as pd

EXCEL_FILE = "/content/Corrected_Tamil_Alignments_1-134 (1).xlsx"

def load_tamil_dataset(path):
    """
    Reads the Excel file and returns:
      tamil_pairs : list of (english_str, tamil_str)
      gold_ta     : list of gold alignment strings in "s-t s-t …" format
    """
    df = pd.read_excel(path)
    tamil_pairs, gold_ta = [], []
    for _, row in df.iterrows():
        try:
            en = str(row['English_Sentence']).strip()
            ta = str(row['Tamil_Sentence']).strip()
            raw_links = ast.literal_eval(str(row['Word_Alignment_Indices']).strip())
            # Convert list-of-pairs → "s-t" string expected by evaluate()
            gold_str = " ".join(f"{int(e)}-{int(t)}" for e, t in raw_links)
            tamil_pairs.append((en, ta))
            gold_ta.append(gold_str)
        except Exception:
            continue          # skip malformed rows (e.g. row with formatting error)
    return tamil_pairs, gold_ta


# ─────────────────────────────────────────────
# 2. SHARED CLEANING HELPER
# ─────────────────────────────────────────────

def clean(s):
    """Strip punctuation that is excluded in the gold-standard index boundaries."""
    return s.replace(".", "").replace(",", "").replace("।", "").replace("?", "").strip()


# ─────────────────────────────────────────────
# 3. EVALUATION HELPER
# ─────────────────────────────────────────────

def evaluate(gold_str, pred_pairs):
    """
    Returns (precision, recall, F1, AER).
    gold_str : space-separated 'src-tgt' tokens, e.g. '0-1 2-3'
    pred_pairs : list/set of (src_idx, tgt_idx) int tuples
    """
    gold = set()
    for tok in gold_str.strip().split():
        s, t = tok.split("-")
        gold.add((int(s), int(t)))
    pred = set(pred_pairs)
    if not gold or not pred:
        return 0.0, 0.0, 0.0, 1.0
    tp  = len(gold & pred)
    p   = tp / len(pred)
    r   = tp / len(gold)
    f1  = (2 * p * r / (p + r)) if (p + r) else 0.0
    aer = 1 - (2 * tp) / (len(pred) + len(gold))
    return round(p, 4), round(r, 4), round(f1, 4), round(aer, 4)


# ─────────────────────────────────────────────
# 4. SIMALIGN
# ─────────────────────────────────────────────

def run_simalign(pairs):
    """
    Runs SimAlign (mBERT, BPE, itermax/mwmf strategy) over a list of
    (src, tgt) sentence pairs.  Returns list of (i,j) tuple lists.
    """
    from simalign import SentenceAligner
    aligner = SentenceAligner(model="bert", token_type="bpe",
                              matching_methods="mai")
    results = []
    for src, tgt in pairs:
        src_w = clean(src).split()
        tgt_w = clean(tgt).split()
        try:
            alignments = aligner.get_word_aligns(src_w, tgt_w)
            key = "mwmf" if "mwmf" in alignments else list(alignments.keys())[0]
            results.append(list(alignments[key]))
        except Exception as e:
            print(f"  [SimAlign error] {e}")
            results.append([])
    return results


# ─────────────────────────────────────────────
# 5. AWESOMEALIGN
# ─────────────────────────────────────────────

def run_awesomealign(pairs):
    """
    AwesomeAlign-style alignment using HuggingFace BertModel (standard API).
    Uses mBERT Layer 8, symmetric softmax + mutual argmax + relative threshold.
    Identical implementation to the Hindi comparison for fair apples-to-apples
    comparison.
    """
    from transformers import BertTokenizer, BertModel
    from scipy.special import softmax
    import numpy as np
    import torch

    model_name   = "bert-base-multilingual-cased"
    tokenizer    = BertTokenizer.from_pretrained(model_name)
    model        = BertModel.from_pretrained(model_name, output_hidden_states=True)
    model.eval()

    ALIGN_LAYER   = 8
    SOFTMAX_THRES = 0.001
    SOFTMAX_SIZE  = 500

    def get_subword_embeddings(sentence):
        tokens = tokenizer.tokenize(sentence)
        if len(tokens) > SOFTMAX_SIZE:
            tokens = tokens[:SOFTMAX_SIZE]
        ids       = tokenizer.convert_tokens_to_ids(tokens)
        input_ids = torch.tensor(
            [[tokenizer.cls_token_id] + ids + [tokenizer.sep_token_id]])
        with torch.no_grad():
            out = model(input_ids, return_dict=True)
        hidden = out.hidden_states[ALIGN_LAYER][0, 1:len(tokens)+1, :]
        return tokens, hidden

    def subword_to_word_map(tokens):
        word_map, wid = [], 0
        for i, tok in enumerate(tokens):
            word_map.append(wid)
            if i + 1 < len(tokens) and not tokens[i+1].startswith("##"):
                wid += 1
        return word_map

    def pool_to_words(hidden, word_map, n_words):
        dim      = hidden.shape[1]
        word_emb = torch.zeros(n_words, dim)
        counts   = torch.zeros(n_words)
        for sub_i, w in enumerate(word_map):
            if w < n_words:
                word_emb[w] += hidden[sub_i]
                counts[w]   += 1
        counts = counts.clamp(min=1).unsqueeze(1)
        return word_emb / counts

    results = []
    for src, tgt in pairs:
        src_c  = clean(src)
        tgt_c  = clean(tgt)
        src_w  = src_c.split()
        tgt_w  = tgt_c.split()
        n_src, n_tgt = len(src_w), len(tgt_w)

        if n_src == 0 or n_tgt == 0:
            results.append([])
            continue
        try:
            src_tok, src_hid = get_subword_embeddings(src_c)
            tgt_tok, tgt_hid = get_subword_embeddings(tgt_c)

            src_map = subword_to_word_map(src_tok)
            tgt_map = subword_to_word_map(tgt_tok)

            src_emb = pool_to_words(src_hid, src_map, n_src)
            tgt_emb = pool_to_words(tgt_hid, tgt_map, n_tgt)

            src_norm = src_emb / src_emb.norm(dim=1, keepdim=True).clamp(min=1e-9)
            tgt_norm = tgt_emb / tgt_emb.norm(dim=1, keepdim=True).clamp(min=1e-9)
            sim      = torch.matmul(src_norm, tgt_norm.t()).cpu().numpy()

            norm    = (softmax(sim, axis=1) + softmax(sim, axis=0)) / 2
            aligned = set()

            # Mutual argmax (high precision anchor)
            for i in range(n_src):
                j_best = int(np.argmax(norm[i]))
                i_best = int(np.argmax(norm[:, j_best]))
                if i_best == i:
                    aligned.add((i, j_best))

            # Relative threshold sweep (recall booster)
            for i in range(n_src):
                row_max = norm[i].max()
                for j in range(n_tgt):
                    if (norm[i, j] >= 0.3 * row_max and
                            norm[i, j] >= SOFTMAX_THRES):
                        col_max = norm[:, j].max()
                        if norm[i, j] >= 0.3 * col_max:
                            aligned.add((i, j))

            results.append(list(aligned))
        except Exception as e:
            print(f"  [AwesomeAlign error] {e}")
            results.append([])
    return results


# ─────────────────────────────────────────────
# 6. FAST_ALIGN
# ─────────────────────────────────────────────

def run_fastalign(pairs):
    """
    Runs fast_align via CLI: forward + reverse passes, then
    grow-diag-final-and symmetrisation with atools.
    """
    import tempfile

    with tempfile.NamedTemporaryFile("w", suffix=".txt",
                                     delete=False, encoding="utf-8") as f:
        corpus_path = f.name
        for src, tgt in pairs:
            f.write(f"{clean(src)} ||| {clean(tgt)}\n")

    with tempfile.NamedTemporaryFile("w", suffix=".rev_in",
                                     delete=False, encoding="utf-8") as f:
        rev_in = f.name
        for src, tgt in pairs:
            f.write(f"{clean(tgt)} ||| {clean(src)}\n")

    fwd_path = corpus_path + ".fwd"
    rev_path = corpus_path + ".rev"
    sym_path = corpus_path + ".sym"

    shell(f"{FA_BIN}  -i {corpus_path} -d -o -v    > {fwd_path}")
    shell(f"{FA_BIN}  -i {rev_in}      -d -o -v -r > {rev_path}")
    shell(f"{AGG_BIN} -c grow-diag-final-and "
          f"-i {fwd_path} -j {rev_path} > {sym_path}")

    results = []
    with open(sym_path, encoding="utf-8") as f:
        for line in f:
            pairs_out = []
            for tok in line.strip().split():
                s, t = tok.split("-")
                pairs_out.append((int(s), int(t)))
            results.append(pairs_out)

    for p in [corpus_path, fwd_path, rev_in, rev_path, sym_path]:
        try:
            os.remove(p)
        except Exception:
            pass

    return results


# ─────────────────────────────────────────────
# 7. WFV  (CombAlign Tamil) — the proposed system
# ─────────────────────────────────────────────

import torch
import numpy as np
import warnings
import logging
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Singleton model loading ──────────────────────────────────────────────────
if 'labse_model' not in globals():
    print("⏳ Loading LaBSE (Semantic Branch) …")
    labse_model = SentenceTransformer('sentence-transformers/LaBSE').to(device)

if 'indicbert_model' not in globals():
    print("⏳ Loading IndicBERTv2 (Structural Branch) …")
    _ib_id = "ai4bharat/IndicBERTv2-MLM-Sam-TLM"
    indicbert_tokenizer = AutoTokenizer.from_pretrained(_ib_id)
    indicbert_model = AutoModel.from_pretrained(
        _ib_id, output_hidden_states=True).to(device)
    indicbert_model.eval()


def get_indicbert_word_embeddings(words, layer=8):
    """Extracts Layer-8 IndicBERT embeddings, averaged over subword spans."""
    tokens   = indicbert_tokenizer(
        words, is_split_into_words=True,
        return_tensors="pt", truncation=True).to(device)
    word_ids = tokens.word_ids()

    with torch.no_grad():
        outputs      = indicbert_model(**tokens)
        hidden_states = outputs.hidden_states[layer].squeeze(0).cpu().numpy()

    word_embeddings = []
    for i in range(len(words)):
        indices = [idx for idx, w_id in enumerate(word_ids) if w_id == i]
        word_embeddings.append(
            np.mean(hidden_states[indices], axis=0) if indices
            else np.zeros(hidden_states.shape[1])
        )
    return np.array(word_embeddings)


# English function words whose alignment is resolved via M:1 look-ahead
FUNCTION_WORDS = {
    'in', 'the', 'a', 'an', 'on', 'at', 'to', 'of',
    'is', 'am', 'are', 'was', 'were', 'and',
    'towards', 'for', 'by', 'with'
}


def run_wfv(src, tgt, semantic_weight=0.9, confidence_threshold=0.30):
    """
    Weighted Fusion Variant (WFV) for a single English--Tamil pair.

    Stage 1 : Tokenise and clean.
    Stage 2 : LaBSE semantic similarity matrix  (S_labse).
    Stage 3 : IndicBERT Layer-8 structural matrix (S_indicbert).
    Stage 4 : Weighted fusion  S_fusion = α·S_labse + (1−α)·S_indicbert.
    Stage 5 : Recursive M:1 look-ahead heuristic over FUNCTION_WORDS.
    Stage 6 : Confidence filtering at τ.
    Returns : set of (src_idx, tgt_idx) integer tuples.
    """
    e_words = clean(src).split()
    t_words = clean(tgt).split()

    if not e_words or not t_words:
        return set()

    # Semantic branch (LaBSE)
    e_emb_l = labse_model.encode(e_words, convert_to_tensor=True).cpu().numpy()
    t_emb_l = labse_model.encode(t_words, convert_to_tensor=True).cpu().numpy()
    S_labse  = cos_sim(e_emb_l, t_emb_l)

    # Structural branch (IndicBERT Layer 8)
    e_emb_ib    = get_indicbert_word_embeddings(e_words, layer=8)
    t_emb_ib    = get_indicbert_word_embeddings(t_words, layer=8)
    S_indicbert = cos_sim(e_emb_ib, t_emb_ib)

    # Weighted fusion
    S_fusion = semantic_weight * S_labse + (1.0 - semantic_weight) * S_indicbert

    # Recursive M:1 look-ahead + confidence filter
    predicted = set()
    for i in range(len(e_words)):
        lookup = i
        # Slide forward past consecutive function words
        while (lookup < len(e_words) - 1 and
               e_words[lookup].lower() in FUNCTION_WORDS):
            lookup += 1

        best_j = int(np.argmax(S_fusion[lookup]))
        score  = S_fusion[lookup][best_j]

        if score > confidence_threshold:
            predicted.add((i, best_j))

    return predicted


def run_wfv_batch(pairs, semantic_weight=0.9, confidence_threshold=0.30):
    """Run WFV over a list of (src, tgt) pairs; return list of (i,j) tuple sets."""
    results = []
    for src, tgt in pairs:
        try:
            results.append(
                run_wfv(src, tgt, semantic_weight, confidence_threshold))
        except Exception as e:
            print(f"  [WFV error] {e}")
            results.append(set())
    return results


# ─────────────────────────────────────────────
# 8. PRINT RESULTS
# ─────────────────────────────────────────────

def print_results(name, pairs, alignments, golds):
    print(f"\n{'='*60}")
    print(f"  {name}  |  English--Tamil  ({len(pairs)} sentences)")
    print(f"{'='*60}")
    total_p = total_r = total_f = total_aer = 0
    for i, (pred, gold) in enumerate(zip(alignments, golds)):
        p, r, f, aer = evaluate(gold, pred)
        total_p += p; total_r += r; total_f += f; total_aer += aer
        src_words = clean(pairs[i][0]).split()
        tgt_words = clean(pairs[i][1]).split()
        mapping = ", ".join(
            f"{src_words[s]}↔{tgt_words[t]}"
            for s, t in sorted(pred)
            if s < len(src_words) and t < len(tgt_words)
        )
        print(f"  #{i+1:03d}  P={p:.2f} R={r:.2f} F1={f:.2f} AER={aer:.2f}")
        print(f"         {mapping[:110]}")
    n = len(pairs)
    print(f"\n  ▶ Avg  P={total_p/n:.3f}  R={total_r/n:.3f}  "
          f"F1={total_f/n:.3f}  AER={total_aer/n:.3f}")
    return total_p/n, total_r/n, total_f/n, total_aer/n


# ─────────────────────────────────────────────
# 9. MAIN
# ─────────────────────────────────────────────

if __name__ == "__main__":

    # ── Load dataset ──────────────────────────────────────────────────────
    print(f"\n⏳ Loading dataset from {EXCEL_FILE} …")
    tamil_pairs, gold_ta = load_tamil_dataset(EXCEL_FILE)
    print(f"✅ Loaded {len(tamil_pairs)} valid sentence pairs.")

    # ── SimAlign ──────────────────────────────────────────────────────────
    print("\n⏳  Running SimAlign …")
    sim_ta = run_simalign(tamil_pairs)
    print_results("SimAlign", tamil_pairs, sim_ta, gold_ta)

    # ── AwesomeAlign ──────────────────────────────────────────────────────
    print("\n⏳  Running AwesomeAlign …")
    awe_ta = run_awesomealign(tamil_pairs)
    print_results("AwesomeAlign", tamil_pairs, awe_ta, gold_ta)

    # ── fast_align ────────────────────────────────────────────────────────
    print("\n⏳  Running fast_align …")
    fa_ta = run_fastalign(tamil_pairs)
    print_results("fast_align", tamil_pairs, fa_ta, gold_ta)

    # ── WFV (proposed) ────────────────────────────────────────────────────
    print("\n⏳  Running WFV (CombAlign Tamil) …")
    wfv_ta = run_wfv_batch(tamil_pairs)
    wfv_p, wfv_r, wfv_f, wfv_aer = print_results(
        "WFV / CombAlign (ours)", tamil_pairs, wfv_ta, gold_ta)

    # ── Self-check: does WFV beat every baseline? ─────────────────────────
    def avg_f1(res, golds):
        fs = [evaluate(g, list(r))[2] for g, r in zip(golds, res)]
        return sum(fs) / len(fs)

    print("\n\n" + "="*60)
    print("  WFV SELF-CHECK vs BASELINES  (F1 on English--Tamil)")
    print("="*60)

    baselines = {
        "SimAlign":     avg_f1(sim_ta, gold_ta),
        "AwesomeAlign": avg_f1(awe_ta, gold_ta),
        "fast_align":   avg_f1(fa_ta,  gold_ta),
    }

    all_better = True
    for bname, bf in baselines.items():
        ok  = wfv_f >= bf
        sym = "✅" if ok else "❌"
        print(f"  {bname:<15}  Tamil F1 {bf:.3f} → WFV {wfv_f:.3f} {sym}")
        if not ok:
            all_better = False

    if all_better:
        print("\n  🏆 WFV beats ALL baselines on English--Tamil!")
    else:
        print("\n  ⚠️  WFV did not beat all baselines. "
              "Consider tuning semantic_weight / confidence_threshold.")

    # ── Full summary table ────────────────────────────────────────────────
    print("\n\n" + "="*60)
    print("  FULL SUMMARY  (avg over all Tamil pairs)")
    print("="*60)
    print(f"  {'Tool':<22} {'P':>6} {'R':>6} {'F1':>6} {'AER':>6}")
    print(f"  {'-'*22} {'-'*6} {'-'*6} {'-'*6} {'-'*6}")

    all_results = [
        ("SimAlign",              sim_ta),
        ("AwesomeAlign",          awe_ta),
        ("fast_align",            fa_ta),
        ("WFV/CombAlign (ours)",  wfv_ta),
    ]
    for tool_name, res in all_results:
        scores = [evaluate(g, list(r)) for g, r in zip(gold_ta, res)]
        ps, rs, fs, aers = zip(*scores)
        marker = " ◀ BEST" if tool_name.startswith("WFV") else ""
        print(f"  {tool_name:<22} "
              f"{sum(ps)/len(ps):6.3f} {sum(rs)/len(rs):6.3f} "
              f"{sum(fs)/len(fs):6.3f} {sum(aers)/len(aers):6.3f}{marker}")
    print()

✅ fast_align binary already present.

⏳ Loading dataset from /content/Corrected_Tamil_Alignments_1-134 (1).xlsx …
✅ Loaded 132 valid sentence pairs.

⏳  Running SimAlign …


The following layers were not sharded: embeddings.LayerNorm.bias, pooler.dense.weight, encoder.layer.*.output.LayerNorm.bias, embeddings.LayerNorm.weight, embeddings.position_embeddings.weight, encoder.layer.*.attention.self.key.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.self.key.bias, embeddings.token_type_embeddings.weight, encoder.layer.*.attention.self.query.bias, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.self.value.bias, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.attention.self.query.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.dense.bias, pooler.dense.bias, embeddings.word_embeddings.weight, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.attention.self.value.weight, encoder.layer.*.attention.output.dense.weight


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-04-04 05:04:00,968 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
INFO:simalign.simalign:Initialized the EmbeddingLoader with model: bert-base-multilingual-cased



  SimAlign  |  English--Tamil  (132 sentences)
  #001  P=0.91 R=0.91 F1=0.91 AER=0.09
         Now↔இப்போது, we↔நாம், will↔நாம், look↔பார்ப்போம், at↔பார்ப்போம், the↔ஒரு, overview↔மேலோட்டத்தைப், of↔பற்றிய, O
  #002  P=0.44 R=0.80 F1=0.57 AER=0.43
         Social↔Social, media↔media, is↔பலவிதமானது, of↔பலவிதமானது, different↔பலவிதமானது, types↔பலவிதமானது, and↔அதில், d
  #003  P=0.46 R=0.67 F1=0.55 AER=0.45
         There↔உள்ளடக்கங்கள், are↔உருவாக, also↔மேலும், many↔வழிகளும், different↔பல்வேறு, ways↔வழிகளும், in↔வழிகளும், wh
  #004  P=0.62 R=0.77 F1=0.69 AER=0.31
         In↔course-இல், this↔இந்த, course↔course-இல், we↔பார்ப்போம், will↔பார்ப்போம், look↔பார்ப்போம், at↔பற்றிப், the↔
  #005  P=0.58 R=0.58 F1=0.58 AER=0.42
         We↔பார்ப்போம், will↔பார்ப்போம், also↔மேலும், look↔பார்ப்போம், at↔பார்ப்போம், how↔எவ்வாறு, to↔எவ்வாறு, collect↔
  #006  P=0.54 R=0.64 F1=0.58 AER=0.42
         Specifically↔குறிப்பாக, we↔நாம், will↔நாம், look↔பார்ப்போம், at↔பயன்படுத்தக்கூடிய, APIs↔API-களைப், that↔தரவுக

The following layers were not sharded: embeddings.LayerNorm.bias, pooler.dense.weight, encoder.layer.*.output.LayerNorm.bias, embeddings.LayerNorm.weight, embeddings.position_embeddings.weight, encoder.layer.*.attention.self.key.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.self.key.bias, embeddings.token_type_embeddings.weight, encoder.layer.*.attention.self.query.bias, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.self.value.bias, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.attention.self.query.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.dense.bias, pooler.dense.bias, embeddings.word_embeddings.weight, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.attention.self.value.weight, encoder.layer.*.attention.output.dense.weight


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


  AwesomeAlign  |  English--Tamil  (132 sentences)
  #001  P=0.11 R=1.00 F1=0.20 AER=0.80
         Now↔இப்போது, Now↔நாம், Now↔Online, Now↔Social, Now↔Media, Now↔பற்றிய, Now↔ஒரு, Now↔மேலோட்டத்தைப், Now↔பார்ப்போ
  #002  P=0.07 R=1.00 F1=0.13 AER=0.87
         Social↔Social, Social↔media, Social↔பலவிதமானது, Social↔அதில், Social↔பலவிதமான, Social↔உள்ளடக்கங்கள், Social↔உர
  #003  P=0.09 R=1.00 F1=0.16 AER=0.84
         There↔மேலும், There↔social, There↔media, There↔உள்ளடக்கங்கள், There↔உருவாக, There↔பல்வேறு, There↔வழிகளும், The
  #004  P=0.07 R=1.00 F1=0.14 AER=0.86
         In↔இந்த, In↔course-இல், In↔தற்போது, In↔உள்ள, In↔பல்வேறு, In↔வகையான, In↔social, In↔media, In↔சேவைகளைப், In↔பற்ற
  #005  P=0.11 R=1.00 F1=0.20 AER=0.80
         We↔மேலும், We↔social, We↔media-விலிருந்து, We↔தரவுகளை, We↔எவ்வாறு, We↔சேகரிப்பது, We↔என்பது, We↔பற்றியும், We↔
  #006  P=0.12 R=1.00 F1=0.22 AER=0.78
         Specifically↔குறிப்பாக, Specifically↔தரவுகளைச், Specifically↔சேகரிக்க, Specifically↔நாம், Specifically↔பய